In [ ]:
import numpy as np
import time

def sketch_and_project(A, b, num_iterations, block_size):
    n, m = A.shape # m is the number of cols of A
    # Set the initial guess x0 = 0
    x = np.zeros(m)
    
    for t in range(num_iterations):
        # Randomly choose a subset B of indices to pick out rows and columns
        B = np.random.choice(n, size=block_size, replace = False)
        # Now we need to extract the relevant submatrix and subvectors
        A_BB = A[np.ix_(B,B)] # This is matrix A with rows B and columns B only
        A_B = A[B,:] # This is matrix A with rows B but all columns
        b_B = b[B] # This is the vector b but with only rows B
        #Now we compute the residual of the equations
        r_B = A_B @ x - b_B
        #Solve the small linear system to get the correction vector d
        d_B = np.linalg.solve(A_BB, r_B)
        #Update the parts of the vector x in B
        x[B] -= d_B
    return x

# Test 1: On a small matrix
A = np.array([
    [2.0, 1.0, 0.0],
    [1.0, 2.0, 1.0],
    [0.0, 1.0, 2.0]
])

b = np.array([1.0, 2.0, 3.0])

x_approx = sketch_and_project(
    A, b,
    num_iterations=500,
    block_size=2
)

print("Sketch-and-project solution 1:")
print(x_approx)

#Test 2: On a large matrix
np.random.seed(0) # for reproducibility

n = 5000 # size of the system
block_size = 20
num_iterations = 2000

# Step 1: true solution
x_true = np.random.randn(n)

# Step 2: construct a symmetric positive definite matrix A = M^T M + λI
M = np.random.randn(n, n)
A = M.T @ M + 10 * np.eye(n) # We use λ = 10 here to improve conditioning, using λ = 0.1 the matrix takes very long to converge

# Step 3: RHS
b = A @ x_true

start = time.perf_counter()

x_approx = sketch_and_project(
    A, b,
    num_iterations=num_iterations,
    block_size=block_size
)

end = time.perf_counter()
SAP_time = end - start

start_1 = time.perf_counter()

x_direct = np.linalg.solve(A, b)

end_1 = time.perf_counter()

GE_time = end_1 - start_1

solution_error = np.linalg.norm(x_approx - x_true)
print("Solution error:", solution_error)
rel_error = np.linalg.norm(x_approx - x_true) / np.linalg.norm(x_true)
print("Relative error:", rel_error)
print("Sketch-and-project time (seconds):", SAP_time)
print("Gaussian elimination time (seconds):", GE_time)

# Running the code above, we can solve the simple small matrix, getting the solution x = [5, 0, 1.5].
# Then solving a large system with a large matrix A, with n=1000, num_iterations=10000, block size = 50 we get good agreement between actual solution and the SAP solution. The results are:

# Solution error: 0.025848151018749183
# Relative error: 0.0008272593836802964
# Sketch-and-project time (seconds): 2.2151416929991683
# Gaussian elimination time (seconds): 0.016270128995529376

# But in such a case, we note that SAP is actually slower than GE! This is due to the large number of iterations, large block size, and also n is not large enough for SAP to truly shine.
# So this is where we see the flaws of SAP - it depends on well conditioned matrices that can convege fast enough.
# We run the code again with n=5000, num_iterations=2000, and block_size=20. Doing so, we get the following results:

# Solution error: 45.303167214777
# Relative error: 0.6501451919055787
# Sketch-and-project time (seconds): 0.2626515379961347
# Gaussian elimination time (seconds): 1.218804659001762

# Note that the solution error is larger in this case. So it will be an issue of balancing accuracy with speed, and well-conditioned matrices will reduce the need for big tradeoffs.

Sketch-and-project solution 1:
[5.00000000e-01 6.84195572e-17 1.50000000e+00]
Solution error: 37.24474910261952
Relative error: 0.534498933330635
Sketch-and-project time (seconds): 0.5341122089885175
Gaussian elimination time (seconds): 1.255646894001984


In [ ]:
import jax
import jax.numpy as jnp
from jax import lax, random, jit
import numpy as np # Used only for data generation
import time
from functools import partial

# 1. Enable 64-bit precision (Crucial for linear solvers)
jax.config.update("jax_enable_x64", True)

@partial(jit, static_argnames=['block_size', 'num_iterations'])
def sketch_and_project_jax(A, b, key, num_iterations, block_size):
    """
    Solves Ax=b using Randomized Block Kaczmarz with JAX JIT compilation.
    """
    n, m = A.shape
    x0 = jnp.zeros(m)

    # We define the body of the loop to be compiled
    def body_fun(i, val):
        x, current_key = val

        # Split key for randomness
        current_key, subkey = random.split(current_key)

        # 1. Randomly sample indices (rows)
        # Note: replace=False is computationally heavier in JAX than replace=True.
        # For very large N, replace=True is fine, but we stick to False for correctness.
        idx = random.choice(subkey, n, shape=(block_size,), replace=False)

        # 2. Extract Submatrices (Slicing)
        A_B = A[idx, :]
        b_B = b[idx]

        # 3. Compute Residual
        r = A_B @ x - b_B

        # 4. Solve the Gram system: (A_B @ A_B.T) z = r
        # We add a tiny jitter to diagonal for numerical stability
        gram = A_B @ A_B.T
        # gram = gram + 1e-10 * jnp.eye(block_size) # Uncomment if system is very ill-conditioned

        z = jnp.linalg.solve(gram, r)

        # 5. Update x
        x_new = x - A_B.T @ z

        return (x_new, current_key)

    # Run the loop efficiently using XLA
    # val is the tuple (initial_x, initial_key)
    final_x, _ = lax.fori_loop(0, num_iterations, body_fun, (x0, key))

    return final_x

# --- Test Setup ---

def run_benchmark():
    print("Generating data...")
    np.random.seed(0)

    # Size Parameters
    n = 35000
    block_size = 500
    num_iterations = 5000

    # Generate Data (using NumPy for setup)
    x_true = np.random.randn(n)
    M = np.random.randn(n, n)
    # Make A Symmetric Positive Definite (easier to solve)
    A_np = M.T @ M + 10 * np.eye(n)
    b_np = A_np @ x_true

    # Convert to JAX Arrays (device transfer)
    A_jax = jnp.array(A_np)
    b_jax = jnp.array(b_np)
    key = random.PRNGKey(42)

    print(f"System size: {n}x{n}")
    print(f"Iterations: {num_iterations}, Block size: {block_size}")
    print("-" * 30)

    # --- 1. JAX Sketch and Project ---

    print("Compiling JAX Kaczmarz...")
    # Trigger JIT compilation with a warm-up run (not timed)
    _ = sketch_and_project_jax(A_jax, b_jax, key, 10, block_size).block_until_ready()
    print("Compilation done. Running benchmark...")

    start_sap = time.perf_counter()

    # Actual Run
    x_approx = sketch_and_project_jax(A_jax, b_jax, key, num_iterations, block_size)
    # Important: JAX is asynchronous. We must block until result is ready to time it correctly.
    x_approx.block_until_ready()

    end_sap = time.perf_counter()
    time_sap = end_sap - start_sap

    # --- 2. JAX Direct Solve (Gaussian Elimination/LU) ---

    print("Running JAX Direct Solve (jnp.linalg.solve)...")
    start_direct = time.perf_counter()
    x_direct = jnp.linalg.solve(A_jax, b_jax)
    x_direct.block_until_ready()
    end_direct = time.perf_counter()
    time_direct = end_direct - start_direct

    # --- Results ---

    # Move results back to CPU for printing
    err = jnp.linalg.norm(x_approx - jnp.array(x_true))
    rel_err = err / jnp.linalg.norm(jnp.array(x_true))

    print("-" * 30)
    print(f"Sketch-and-Project Time:  {time_sap:.5f} s")
    print(f"Direct Solve Time:        {time_direct:.5f} s")
    print(f"SAP Relative Error:       {rel_err:.6f}")

    if time_sap < time_direct:
        print("\nResult: SAP was FASTER than Direct Solve.")
    else:
        print("\nResult: SAP was SLOWER than Direct Solve.")

if __name__ == "__main__":
    run_benchmark()

Generating data...
System size: 35000x35000
Iterations: 5000, Block size: 500
------------------------------
Compiling JAX Kaczmarz...
Compilation done. Running benchmark...
Running JAX Direct Solve (jnp.linalg.solve)...
------------------------------
Sketch-and-Project Time:  15.65360 s
Direct Solve Time:        2.02157 s
SAP Relative Error:       0.442769

Result: SAP was SLOWER than Direct Solve.
